In [4]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [6]:

# 1. Connect to BigQuery
client = bigquery.Client()

# 2. Read RFM model
query = """
SELECT *
FROM `olist-data-pipeline-507001.olist_mart.fct_customer_rfm`
"""

rfm = client.query(query).to_dataframe()

# 3. Inspect
print("Shape:", rfm.shape)
display(rfm.head())

# 4. Export CSV
rfm.to_csv("../outputs/eda/rfm_customer.csv", index=False)
print("Saved: rfm_customer.csv")


/home/rachel/miniconda3/envs/dagster/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Shape: (94990, 11)


,customer_key,recency,frequency,monetary,recency_score,frequency_score,monetary_percentile,monetary_score,rfm_code,rfm_total_score,customer_segment
0,a6a3e9167b88bd2d28c9a4b34b606642,56,1,14.900000000,4,1,0.023592,1,411,6,New Customers
1,f300a666ea87e7df5ca308533139b01b,57,1,18.900000000,4,1,0.042121,1,411,6,New Customers
2,52d6883c8c166e738c8d86ec5abd3ae3,60,1,19.900000000,4,1,0.050511,1,411,6,New Customers
3,b68175d8da0aed5eb128b5b0d8cb0b8b,59,1,22.830000000,4,1,0.074545,1,411,6,New Customers
4,282ce2fc6f789ce4f3706575885cf50e,52,1,24.990000000,4,1,0.089874,1,411,6,New Customers


Saved: rfm_customer.csv


In [7]:
#Read the CSV file back into a DataFrame
rfm = pd.read_csv("../outputs/eda/rfm_customer.csv")

#5. Inspect basic information about the DataFrame
print("Shape:", rfm.shape)

print("\nColumns:")
print(rfm.columns.tolist())

print("\nData types:")
print(rfm.dtypes)



Shape: (94990, 11)

Columns:
['customer_key', 'recency', 'frequency', 'monetary', 'recency_score', 'frequency_score', 'monetary_percentile', 'monetary_score', 'rfm_code', 'rfm_total_score', 'customer_segment']

Data types:
customer_key            object
recency                  int64
frequency                int64
monetary               float64
recency_score            int64
frequency_score          int64
monetary_percentile    float64
monetary_score           int64
rfm_code                 int64
rfm_total_score          int64
customer_segment        object
dtype: object


In [27]:
#Inspect RFM distribution
rfm_summary = (
    rfm[['recency', 'frequency', 'monetary']]
    .describe(percentiles=[.50, .75, .90, .95])
    .T
    .reset_index()
    .rename(columns={'index': 'metric'})
)

display(rfm_summary)

,metric,count,mean,std,min,50%,75%,90%,95%,max
0,recency,94990.0,288.349479,153.000542,45.00,269.00,397.0,517.000,570.0,774.0
1,frequency,94990.0,1.033867,0.210826,1.00,1.00,1.0,1.000,1.0,16.0
2,monetary,94983.0,142.071747,216.074999,0.85,89.89,155.0,281.592,420.0,13440.0


RFM Scoring Methodology

The RFM distribution is highly skewed:

Recency is widely distributed, with a median of 269 days and a maximum of 774 days.
Frequency is heavily dominated by one-time purchasers, with at least 95% of customers having only one order.
Monetary value has a strong right-skewed distribution, with a median of 89.89 compared with a maximum of 13,440.

Because Recency and Monetary do not have clear natural business boundaries for defining score intervals, percentile-based scoring is used. Percentile scoring evaluates each customer's position relative to the overall customer population and reduces the influence of extreme values.

For example, Monetary scoring identifies the top 20% of customers based on spending, regardless of how extreme the highest values are.

Frequency uses business-defined bands instead because its distribution is highly concentrated at one order. Percentile scoring would not provide meaningful differentiation between the large number of one-time purchasers.

| Dimension         | Scoring method         | Reason                                                     |
| ----------------- | ---------------------- | ---------------------------------------------------------- |
| **Recency (R)**   | Percentile             | Widely distributed and skewed; no natural cutoff intervals |
| **Frequency (F)** | Business-defined bands | At least 95% of customers have only one order              |
| **Monetary (M)**  | Percentile             | Strong right skew and extreme high-value outliers          |


| Dimension     |  Score  |
| ------------- | --------|
| **Recency**   |  1–5    |
| **Frequency** |  1–4    |
| **Monetary**  |  1–5    |


In summary: Percentile-based scoring converts relative customer performance into comparable 1–5 scores for Recency and Monetary, while Frequency uses business-defined rules to better reflect repeat-purchase behavior.